# Decorators and Generators

Practice notebook covering two core Python concepts:

- **Decorators** — writing a wrapper function that adds behavior before/after a function call, then applying it both manually and with `@` syntax.
- **Generators** — building a generator function with `yield` (`cubed_numbers`), and manually stepping through an iterator with `iter()` and `next()`.

In [ ]:
#Decorators
def new_decorator(original_func):

    def wrap_func(): 
        print("Some extra code, before the orignial function")

        original_func()

        print('Some extra code, after the original function!')
    
    return wrap_func

In [ ]:
def func_need_decorator(): 
    print("I want to be decorated!!")

In [ ]:
decorator_func = new_decorator(func_need_decorator)

In [ ]:
decorator_func()

In [ ]:
@new_decorator
def func_need_decorator(): 
    print("I want to be decorated!!")

In [ ]:
func_need_decorator()

# Generators

In [ ]:
#generators 
def cubed_numbers(): 
    for num in range(10): 
        yield num**3


In [ ]:
for x in cubed_numbers():
    print(x)

In [ ]:
s = "The world is a cold place to be in, if you have to do it alone."

In [ ]:
print(next(s))

In [ ]:
word_iter = iter(s)

In [ ]:
print(next(word_iter))

In [ ]:
print(next(word_iter))

In [ ]:
print(next(word_iter))

## Itertools Module

itertools module has a collection of generators for many common data algorithms. 

**Table 3-2. Some useful itertools functions**

| Function | Description |
|---|---|
| `chain(*iterables)` | Generates a sequence by chaining iterators together. Once elements from the first iterator are exhausted, elements from the next iterator are returned, and so on. |
| `combinations(iterable, k)` | Generates a sequence of all possible k-tuples of elements in the iterable, ignoring order and without replacement (see also the companion function `combinations_with_replacement`). |
| `permutations(iterable, k)` | Generates a sequence of all possible k-tuples of elements in the iterable, respecting order. |
| `groupby(iterable[, keyfunc])` | Generates `(key, sub-iterator)` for each unique key. |
| `product(*iterables, repeat=1)` | Generates the Cartesian product of the input iterables as tuples, similar to a nested for loop. |

*Excerpt from Python for Data Analysis, Wes McKinney. This material may be protected by copyright.*

In [ ]:
import itertools

In [ ]:
def first_letter(x):
    return x[0]

names = ["Alan", "Adam", "Wes", "Will", "Albert", "Steven"]

In [ ]:
for letter, names in itertools.groupby(names, first_letter):
    print(letter, list(names)) #names is a generator

**Data analyst use case for `groupby`:** collapsing consecutive rows that share a key — e.g., grouping consecutive log rows into "sessions," or (as above) grouping a sorted list of names by first letter for a quick frequency breakdown. Note `itertools.groupby` only groups *consecutive* matches, so the input must already be sorted/ordered by the key — for grouping an unsorted column, `pandas.DataFrame.groupby` is almost always the better tool.

### `chain()`

**Use it when:** you have several separate iterables (lists, query results, file reads) that logically belong to one sequence, and you want to loop over/process them as if they were one — without the cost of building a new combined list.

In [ ]:
# Example: Q1 sales came from three separate regional exports (three lists).
# Instead of concatenating them into a new list, chain them into one pass for a total.

q1_west = [1200, 1450, 980]
q1_east = [2200, 1750]
q1_central = [900, 1100, 1600, 800]

all_q1_sales = list(itertools.chain(q1_west, q1_east, q1_central))
print(all_q1_sales)
print("Total Q1 sales:", sum(itertools.chain(q1_west, q1_east, q1_central)))

### `combinations(iterable, k)`

**Use it when:** order doesn't matter and you need every unique unordered pairing/grouping — e.g., checking correlation between every pair of metrics, comparing every pair of A/B test variants, or finding every possible pair of stores to compare regional performance.

In [ ]:
# Example: you have 4 marketing channels and want every unique PAIR to test
# for cross-channel correlation in weekly spend, without duplicate/reversed pairs.

channels = ["email", "social", "paid_search", "affiliate"]

for pair in itertools.combinations(channels, 2):
    print(pair)

### `permutations(iterable, k)`

**Use it when:** order *does* matter — e.g., testing every possible ordering of steps in a checkout funnel, every sequence of ad exposures a user could see, or ranking scenarios where "A then B" is different from "B then A". Grows much faster than `combinations`, so keep `k` and the input small.

In [ ]:
# Example: a 3-page onboarding flow (welcome, profile, tutorial) could be shown
# in any order during an A/B test. Enumerate every possible ordering to test.

pages = ["welcome", "profile", "tutorial"]

for ordering in itertools.permutations(pages):
    print(ordering)

### `product(*iterables, repeat=1)`

**Use it when:** you need every combination of values across two or more independent dimensions — e.g., building a full grid of (region × product × month) to make sure a report has no missing combinations, or generating every scenario for a parameter sweep / pricing sensitivity analysis.

In [ ]:
# Example: you're building a template for a sales report and need a row for
# every (region, product) pair, even ones with zero sales, so nothing is silently missing.

regions = ["North", "South"]
products = ["Widget", "Gadget"]

report_grid = list(itertools.product(regions, products))
for region, product in report_grid:
    print(f"{region:<6} | {product}")